# Trajectory and velocity analysis of scRNAseq COLON data 

This is one in the series of notebooks presenting the data analysis done in the study "Widespread epithelial dedifferentiation in patients with ulcerative colitis". It is also intended as a tutorial in scRNAseq analysis. Please refer to the REAMDE file for the description of different parts of the tutorial and how to follow it.  

## Part 2. Normalization and batch correction

### All dataset

This notebook shows the results and parameter choices for the "All dataset" (Healthy, Ulcerated and Nonulcerated) (see study for details regarding samples constituting the dataset). I am not providing detailed descriptions of each step here. Those can be found in the notebook for the Healthy dataset. Please refer to the file `Healthy_norm.ipynb`

In [1]:
import scanpy as sc
import scvelo as scv

#### 1. Load data

In [2]:
adata = sc.read('all_qc.h5ad')

#### 2. Normalisation

In [3]:
scv.pp.normalize_per_cell(adata, max_proportion_per_cell=0.05)
scv.pp.log1p(adata)
adata.raw = adata

Normalized count data: X, spliced, unspliced.


#### 3. Gene set scores

In [4]:
# Load the list of genes associated with cell cycle.
cell_cycle_genes = [x.strip() for x in open('Extra files/cell_cycle_genes.txt')]
cell_cycle_genes = cell_cycle_genes[1:]
s_genes = cell_cycle_genes[:43]
g2m_genes = cell_cycle_genes[43:]

# Load the list of genes associated with proliferation.
proliferation_genes = [x.strip() for x in open('Extra files/proliferation_genes.txt')]

In [5]:
# Use a generic function for scoring gene set over expression.
sc.tl.score_genes(adata, proliferation_genes, score_name='Proliferation')

In [6]:
# Use function specifically for scoring cell cycle.
sc.tl.score_genes_cell_cycle(adata, s_genes=s_genes, g2m_genes=g2m_genes)

#### 4. Batch correction

In [7]:
sc.pp.highly_variable_genes(adata, n_top_genes=2000, subset=False)

Please notice that when giving the list of samples we mix the samples from different conditions so that the algorithm is more robust to for biological variation between conditions. Having for instance, all healthy samples processed first could results in the algorithm correction being biased towards healthy dataset

In [8]:
corrected = sc.external.pp.mnn_correct(adata[adata.obs['Sample']=='healthy1'], adata[adata.obs['Sample']=='ulcerated1'],
                                       adata[adata.obs['Sample']=='nonulcerated1'], adata[adata.obs['Sample']=='healthy2'],
                                       adata[adata.obs['Sample']=='ulcerated2'], adata[adata.obs['Sample']=='nonulcerated2'],
                                       adata[adata.obs['Sample']=='healthy3'], adata[adata.obs['Sample']=='ulcerated3'],
                                       adata[adata.obs['Sample']=='nonulcerated3'], adata[adata.obs['Sample']=='healthy4'],
                                       adata[adata.obs['Sample']=='ulcerated4'],
                                  var_subset=list(adata.var_names[adata.var['highly_variable']]), k=15, var_adj=True, 
                                  save_raw=True, n_jobs=12)

Performing cosine normalization...
Starting MNN correct iteration. Reference batch: 0
Step 1 of 10: processing batch 1
  Looking for MNNs...
  Computing correction vectors...
  Adjusting variance...
  Applying correction...
Step 2 of 10: processing batch 2
  Looking for MNNs...
  Computing correction vectors...
  Adjusting variance...
  Applying correction...
Step 3 of 10: processing batch 3
  Looking for MNNs...
  Computing correction vectors...
  Adjusting variance...
  Applying correction...
Step 4 of 10: processing batch 4
  Looking for MNNs...
  Computing correction vectors...
  Adjusting variance...
  Applying correction...
Step 5 of 10: processing batch 5
  Looking for MNNs...
  Computing correction vectors...
  Adjusting variance...
  Applying correction...
Step 6 of 10: processing batch 6
  Looking for MNNs...
  Computing correction vectors...
  Adjusting variance...
  Applying correction...
Step 7 of 10: processing batch 7
  Looking for MNNs...
  Computing correction vectors.

In [9]:
adata = corrected[0].copy()

#### 5. Linear regression

In [10]:
sc.pp.regress_out(adata, ['S_score', 'G2M_score'])

... storing 'Sample' as categorical
... storing 'Condition' as categorical
... storing 'phase' as categorical


#### 6. Save data for the next step 

In [11]:
adata.write("all_norm.h5ad")